# Black-Scholes Option Pricing: A Reproducible Student Research Note

A semi-formal Python notebook connecting option pricing theory, implementation, and interpretation for European options.

**Project type:** academic / portfolio research note  
**Primary question:** how do moneyness, maturity, interest rates, dividends, and volatility translate into option prices, Greeks, and implied volatility?  
**Important caveat:** this notebook is an educational model study, not investment advice or a production trading system.

## Abstract

This research note studies the Black-Scholes-Merton framework for European call and put options. It implements closed-form prices, Greeks, put-call parity checks, implied-volatility inversion, and a Monte Carlo sanity check in Python. The aim is not to argue that Black-Scholes fully explains real option markets. Instead, the model is treated as a benchmark: a transparent no-arbitrage framework that links assumptions to prices and risk exposures. The notebook is written from the perspective of a FinTech student building practical fluency in quantitative finance, with emphasis on reproducibility, clear assumptions, and honest limitations.

## Research Aim and Context

As a Singapore-based FinTech student, my practical motivation is to understand the mechanics behind derivative pricing rather than to present a live trading strategy. Singapore is a regional financial centre, so option pricing is relevant not only for trading roles, but also for risk management, structured products, treasury, and quantitative technology work.

This notebook therefore focuses on three student-level research goals:

- Translate the Black-Scholes-Merton formula into tested Python code.
- Explain how prices and Greeks respond to key inputs.
- Show how implied volatility can be recovered from observed option prices.

The contribution is intentionally modest but concrete. A good early quant-finance project should be technically correct, reproducible, and clear about what it does not prove.

## Personal Learning Position

I am approaching Black-Scholes as a learner, not as someone claiming to have discovered a new pricing model. What interests me is that a relatively compact formula can connect probability, calculus, finance, and software implementation. At first glance, the model looks like a formula to memorize. After working through it in Python, it feels more like a framework for thinking: what assumptions are being made, what risks are being isolated, and what information is being compressed into volatility.

My view at this stage is that Black-Scholes is most useful when it is treated with respect but not treated as final truth. It is elegant, but the assumptions are strong. That makes it a good learning tool because every limitation points toward a real market complication: volatility smiles, discrete dividends, jumps, liquidity, and hedging costs.

## 1. Model Setup

Assume the underlying asset follows geometric Brownian motion under the risk-neutral measure:

$$
dS_t = (r - q) S_t dt + \sigma S_t dW_t,
$$

where $S_t$ is the asset price, $r$ is the continuously compounded risk-free rate, $q$ is the continuous dividend yield, $\sigma$ is volatility, and $W_t$ is Brownian motion.

For a European call with strike $K$ and maturity $T$, the Black-Scholes-Merton price is:

$$
C = S_0 e^{-qT}\Phi(d_1) - K e^{-rT}\Phi(d_2),
$$

and for a European put:

$$
P = K e^{-rT}\Phi(-d_2) - S_0 e^{-qT}\Phi(-d_1),
$$

with

$$
d_1 = \frac{\ln(S_0/K) + (r - q + \sigma^2/2)T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}.
$$

### Learning Note

The part I find most interesting is the risk-neutral drift. In normal investing, we usually care about expected return. In Black-Scholes pricing, the expected return of the stock is not directly used in the final formula. Instead, pricing happens under the risk-neutral measure. This initially feels unintuitive, but it makes sense once the option is viewed as something that can be dynamically hedged rather than simply forecasted.

In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src").exists():
        sys.path.insert(0, str(candidate / "src"))
        PROJECT_ROOT = candidate
        break
else:
    PROJECT_ROOT = Path.cwd()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from options_pricing_research.black_scholes import (
    black_scholes_greeks,
    black_scholes_price,
    implied_volatility,
)

plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.float_format = "{:.4f}".format

## Methodology

The notebook uses a deliberately simple experimental design. First, it implements the analytical Black-Scholes-Merton formula in reusable Python functions. Second, it checks the implementation against put-call parity and known benchmark values through unit tests in the repository. Third, it varies one or two inputs at a time to study economic sensitivity. Finally, it inverts the model to recover implied volatility and compares the analytical price with a Monte Carlo estimate.

All examples use hypothetical prices rather than live market data. This keeps the focus on model mechanics and avoids overstating empirical conclusions from a small sample. The same code can later be connected to listed option data for a more advanced empirical paper.

## 2. Baseline Pricing Example

Start with a hypothetical at-the-money European option. The units can be interpreted as dollars, but the formula itself is currency-agnostic. This baseline is intentionally simple: the goal is to create a reference case before changing moneyness, maturity, and volatility.

In [ ]:
spot = 100.0
strike = 100.0
time_to_expiry = 1.0
risk_free_rate = 0.05
volatility = 0.20
dividend_yield = 0.00

call_price = black_scholes_price("call", spot, strike, time_to_expiry, risk_free_rate, volatility, dividend_yield)
put_price = black_scholes_price("put", spot, strike, time_to_expiry, risk_free_rate, volatility, dividend_yield)

pd.DataFrame(
    {
        "quantity": [
            "spot",
            "strike",
            "time_to_expiry_years",
            "risk_free_rate",
            "volatility",
            "dividend_yield",
            "European call price",
            "European put price",
        ],
        "value": [
            spot,
            strike,
            time_to_expiry,
            risk_free_rate,
            volatility,
            dividend_yield,
            call_price,
            put_price,
        ],
    }
)

At this baseline, the call is more expensive than the put because the positive risk-free rate lowers the present value of the strike payment. This is a small point, but it helped me see that option pricing is not only about direction and volatility. Discounting and carry matter too.

## 3. Put-Call Parity

For European options on a dividend-paying asset, put-call parity states:

$$
C - P = S_0 e^{-qT} - K e^{-rT}.
$$

This is a useful implementation check because it follows from no-arbitrage replication rather than from a numerical approximation.

Put-call parity is one of the cleanest ideas in the paper. I like it because it gives a simple way to check whether prices are internally consistent without needing to predict the future. From a learning perspective, it also makes no-arbitrage feel less abstract: two portfolios with the same payoff should not have different prices.

In [ ]:
discounted_spot = spot * np.exp(-dividend_yield * time_to_expiry)
discounted_strike = strike * np.exp(-risk_free_rate * time_to_expiry)
parity_gap = call_price - put_price - (discounted_spot - discounted_strike)

pd.Series(
    {
        "call_minus_put": call_price - put_price,
        "discounted_forward_gap": discounted_spot - discounted_strike,
        "parity_gap": parity_gap,
    }
)

## 4. Price Surface Across Moneyness and Maturity

Option value is strongly shaped by moneyness and time. The grid below varies strike and maturity while holding spot, interest rate, dividend yield, and volatility fixed.

In [ ]:
strikes = np.arange(70, 131, 5, dtype=float)
maturities = np.array([1 / 12, 0.25, 0.5, 1.0, 2.0])

surface = pd.DataFrame(
    {
        maturity: black_scholes_price("call", spot, strikes, maturity, risk_free_rate, volatility, dividend_yield)
        for maturity in maturities
    },
    index=strikes,
)
surface.index.name = "strike"
surface.columns.name = "maturity_years"
surface.head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for maturity in surface.columns:
    ax.plot(surface.index, surface[maturity], marker="o", linewidth=1.8, label=f"T={maturity:.2f}")

ax.set_title("Black-Scholes Call Price by Strike and Maturity")
ax.set_xlabel("Strike")
ax.set_ylabel("Call price")
ax.legend(title="Years")
plt.show()

The surface shows two basic but important effects. First, call prices fall as strike increases because the option becomes less likely to finish in the money. Second, longer maturity generally increases the option's time value because there is more time for the underlying asset to move favourably. This is especially visible for near-the-money options, where uncertainty matters most.

My takeaway from this surface is that option pricing is very visual. A single option price is just one point on a larger shape. Once I see the price across strikes and maturities, the model becomes easier to reason about. This is also why I think plotting is important in quantitative finance: charts can reveal whether the model behaves sensibly before moving to more complex calibration.

## 5. Greeks

The Greeks measure local risk sensitivities. In a trading or risk-management setting, these quantities are often more important than the standalone price.

- Delta: sensitivity to the underlying price.
- Gamma: sensitivity of Delta to the underlying price.
- Vega: sensitivity to volatility, quoted here per 1.00 change in volatility.
- Theta: sensitivity to calendar time, quoted here per year.
- Rho: sensitivity to the risk-free rate, quoted here per 1.00 change in rate.

In [ ]:
pd.DataFrame(
    [
        {"option": "call", **black_scholes_greeks("call", spot, strike, time_to_expiry, risk_free_rate, volatility)},
        {"option": "put", **black_scholes_greeks("put", spot, strike, time_to_expiry, risk_free_rate, volatility)},
    ]
).set_index("option")

In [ ]:
spots = np.linspace(60, 140, 161)
greek_grid = pd.DataFrame(
    [
        {
            "spot": current_spot,
            "price": black_scholes_price("call", current_spot, strike, time_to_expiry, risk_free_rate, volatility),
            **black_scholes_greeks("call", current_spot, strike, time_to_expiry, risk_free_rate, volatility),
        }
        for current_spot in spots
    ]
)

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for ax, column in zip(axes.ravel(), ["price", "delta", "gamma", "vega"], strict=True):
    ax.plot(greek_grid["spot"], greek_grid[column], color="#0f766e", linewidth=2)
    ax.axvline(strike, color="black", linestyle="--", linewidth=1, alpha=0.5)
    ax.set_title(column.title())
    ax.set_xlabel("Spot")

fig.suptitle("Call Price and Greeks as Spot Changes", y=1.02)
fig.tight_layout()
plt.show()

The Greek profiles are useful because they show that an option is not simply a leveraged bet on direction. Around the strike, Gamma and Vega become especially important: small spot movements can change Delta quickly, and uncertainty about volatility has a large impact on price. This is one reason options risk management often focuses on exposure buckets rather than only profit and loss.

The Greeks changed how I think about options. Before studying them, it is tempting to see a call option as mainly a bullish instrument and a put option as mainly a bearish instrument. After computing Delta, Gamma, and Vega, the product feels more multidimensional. A trader or risk manager is not only asking whether the underlying goes up or down; they are also asking how fast Delta changes, how much volatility matters, and how time decay affects the position.

## 6. Implied Volatility

Market participants often quote option prices as implied volatilities. Given an observed option premium, implied volatility is the value of $\sigma$ that makes the model price match the market price.

The next example creates synthetic market prices from a volatility smile and then recovers the implied volatility by inverting the pricing formula.

In [ ]:
smile_strikes = np.arange(75, 126, 5, dtype=float)
moneyness = smile_strikes / spot
true_vols = 0.18 + 0.55 * (np.log(moneyness) ** 2) + 0.06 * np.maximum(1 - moneyness, 0)

market_prices = [
    black_scholes_price("call", spot, k, 0.5, risk_free_rate, sigma)
    for k, sigma in zip(smile_strikes, true_vols, strict=True)
]
recovered_vols = [
    implied_volatility("call", price, spot, k, 0.5, risk_free_rate)
    for k, price in zip(smile_strikes, market_prices, strict=True)
]

iv_table = pd.DataFrame(
    {
        "strike": smile_strikes,
        "market_price": market_prices,
        "true_volatility": true_vols,
        "recovered_implied_volatility": recovered_vols,
    }
)
iv_table.head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(iv_table["strike"], iv_table["true_volatility"], marker="o", label="true volatility")
ax.plot(iv_table["strike"], iv_table["recovered_implied_volatility"], linestyle="--", label="recovered implied volatility")
ax.set_title("Synthetic Implied Volatility Smile")
ax.set_xlabel("Strike")
ax.set_ylabel("Volatility")
ax.legend()
plt.show()

In this synthetic example, the recovered implied volatility matches the volatility used to generate the prices. In live markets, this inversion is useful because option prices are not quoted from one universal volatility assumption. Different strikes and maturities often imply different volatility levels, producing the volatility smile or skew. That pattern is one of the clearest signs that real markets are richer than the constant-volatility Black-Scholes model.

Implied volatility is the concept I find most practically important. It turns a market price into a volatility number, which makes different options easier to compare. My current understanding is that implied volatility is not a direct forecast of future volatility; it is the volatility that makes the model agree with the market price. That difference matters. The market may be pricing demand, supply, event risk, liquidity, and risk premia, not just expected statistical volatility.

## 7. Monte Carlo Sanity Check

The closed-form price can also be compared with a direct risk-neutral simulation. This is not needed for vanilla European options, but it is useful because the same simulation idea extends to payoffs that do not have a closed-form solution.

In [ ]:
rng = np.random.default_rng(42)
n_paths = 250_000
z = rng.standard_normal(n_paths)
terminal_spot = spot * np.exp(
    (risk_free_rate - dividend_yield - 0.5 * volatility**2) * time_to_expiry
    + volatility * np.sqrt(time_to_expiry) * z
)
discounted_payoff = np.exp(-risk_free_rate * time_to_expiry) * np.maximum(terminal_spot - strike, 0.0)

mc_price = discounted_payoff.mean()
mc_standard_error = discounted_payoff.std(ddof=1) / np.sqrt(n_paths)
analytic_price = black_scholes_price("call", spot, strike, time_to_expiry, risk_free_rate, volatility)

pd.Series(
    {
        "analytic_price": analytic_price,
        "monte_carlo_price": mc_price,
        "standard_error": mc_standard_error,
        "z_score_error": (mc_price - analytic_price) / mc_standard_error,
    }
)

The Monte Carlo estimate should be close to the analytical price, with the difference explainable by simulation error. For a vanilla European call, the closed-form solution is faster and more exact. The simulation is included because it is a bridge to more realistic derivatives where closed-form formulas may not exist, such as path-dependent payoffs or models with more complex dynamics.

I included the Monte Carlo check because it connects the formula back to probability. The closed-form solution is elegant, but simulation makes the payoff distribution more tangible. As a student, I find this helpful: analytical formulas show structure, while simulation helps build intuition about where the number comes from.

## 8. Interpretation and Limitations

This notebook demonstrates that Black-Scholes is still useful as a first model because it gives a transparent link between option prices, no-arbitrage replication, and risk-neutral expectation. The closed-form formulas make it possible to compute prices quickly, while the Greeks translate the same model into risk exposures that can be monitored and hedged.

The main result is not that Black-Scholes is always correct. The more realistic conclusion is that it is a benchmark model: useful for intuition, quoting, and sanity checks, but incomplete as a full description of market prices. Real markets exhibit volatility smiles and skews, discrete dividends, jumps, stochastic volatility, transaction costs, early exercise for American options, and liquidity effects.

For a student portfolio, the value of this paper is that it shows the full workflow: mathematical setup, tested implementation, numerical experiments, interpretation, and limitations. The next stage would be to connect the model to real option chain data and study how implied volatility varies across strike and maturity.

## Personal Reflection

My main learning from this project is that Black-Scholes is less about predicting the market and more about organising the problem. It provides a clean baseline where each input has a role: spot and strike define moneyness, maturity defines time value, rates affect discounting, and volatility controls uncertainty. That structure is useful even when the model is wrong in practice.

I also think the model teaches a good habit for quantitative finance: separate what is elegant from what is realistic. The formula is elegant, but real option markets are messy. A student should be able to appreciate both sides. I would not use this notebook to claim that I can price every option accurately. I would use it to show that I understand the baseline model, can implement it carefully, and can explain where it starts to break down.

The biggest open question for me is how much of an observed implied-volatility smile comes from expected future volatility, and how much comes from risk premia, supply and demand, liquidity, and market microstructure. That question feels like the natural bridge from textbook option pricing into real quantitative finance.

## 9. Future Work

This note can be extended in several realistic directions:

- Pull real option-chain data and construct an implied-volatility surface.
- Compare historical volatility with implied volatility before earnings or major macro events.
- Add discrete dividends and compare the impact on equity option prices.
- Implement binomial trees for American options.
- Extend the Monte Carlo section to Asian options or barrier options.
- Calibrate a simple stochastic-volatility model and compare it with Black-Scholes.

The most natural next paper would be: **From Black-Scholes to Volatility Smiles: A Python Study of Implied Volatility Surfaces**.

## References

- Black, F. and Scholes, M. (1973). The Pricing of Options and Corporate Liabilities. *Journal of Political Economy*.
- Merton, R. C. (1973). Theory of Rational Option Pricing. *Bell Journal of Economics and Management Science*.
- Hull, J. C. *Options, Futures, and Other Derivatives*.